# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ADHIRAJ994/Fly-Rank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane confirmation: staying on Lane 2 — Refresh / Content Opportunity Scoring.** No change from Week 1/2/3.

**Note on scope:** this skeleton's header says "Top-20 Review," but the assignment card requires exactly **ten** reviewed rows this week — the top-20 version is the optional `w04_signal_audit.ipynb` sibling. This notebook does the required ten.

## 0. Setup — rebuild the Week-3 feature/label frame

*Not part of the deliverable itself. Same `month=2026-03` slice, same decision window (days 1-15) vs. outcome window (days 16-31) as the data contract — reused so the signal checks and the rule sit on the exact same honest data.*

In [ ]:
import os
import duckdb
import pandas as pd

# Get HF token
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except ImportError:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN not found. Set it as a Colab Secret or environment variable."
    )

# Connect to DuckDB
con = duckdb.connect()

# Install/load extensions
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

# Create Hugging Face secret
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

# -----------------------------
# Locate the March dataset
# -----------------------------
from huggingface_hub import HfApi

api = HfApi()

files = api.list_repo_files(
    "FlyRank/internship-warehouse",
    repo_type="dataset"
)

daily_files = [
    f for f in files
    if "fact_content_daily_performance" in f
    and "sample" not in f
]

march_files = [
    f for f in daily_files
    if "month=2026-03" in f
]

if not march_files:
    raise ValueError("March dataset not found.")

MARCH_GLOB = (
    f"hf://datasets/FlyRank/internship-warehouse/"
    f"{march_files[0].rsplit('/',1)[0]}/*.parquet"
)

print("MARCH_GLOB =", MARCH_GLOB)

In [ ]:
# Decision-window features (days 1-15)
feature_frame = con.sql(f"""
WITH h1 AS (
    SELECT
        client_hash_id,
        content_hash_id,

        AVG(gsc_avg_position) AS avg_position_h1,

        SUM(gsc_impressions) AS impressions_h1,
        SUM(gsc_clicks) AS clicks_h1,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_engaged_sessions
                ELSE NULL
            END
        ) AS engaged_sessions_h1,

        COUNT(*) FILTER (
            WHERE gsc_impressions > 0
        ) AS days_with_impressions_h1

    FROM '{MARCH_GLOB}'

    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-15'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,
    avg_position_h1,
    impressions_h1,
    clicks_h1,

    ROUND(
        clicks_h1::DOUBLE /
        NULLIF(impressions_h1, 0),
        4
    ) AS ctr_h1,

    engaged_sessions_h1,
    days_with_impressions_h1

FROM h1
WHERE days_with_impressions_h1 >= 5
""").df()


# Outcome-window label (days 16-31)
label_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS clicks_h2
FROM '{MARCH_GLOB}'
WHERE report_date BETWEEN DATE '2026-03-16'
                      AND DATE '2026-03-31'
GROUP BY
    client_hash_id,
    content_hash_id
""").df()


# Merge features and labels
dataset = feature_frame.merge(
    label_frame,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Binary target
dataset["decline_label"] = (
    dataset["clicks_h2"] < dataset["clicks_h1"]
).astype(int)

print(f"Rows: {len(dataset):,}")
print(f"Base decline rate: {dataset['decline_label'].mean():.1%}")

dataset.head()

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### 1a. Signal check #1 — CTR vs. position (flag-linked: this is the assumption behind FlyRank's CTR-fix logic)

**Hypothesis:** average CTR falls as position gets worse. If that holds, a page with a *good* position but *low* CTR for that tier is genuinely underperforming — that gap is what the CTR-fix flag is built on.

In [ ]:
position_bins = [0, 3, 10, 20, float('inf')]
position_labels = ['1-3', '4-10', '11-20', '21+']
dataset['position_tier'] = pd.cut(dataset['avg_position_h1'], bins=position_bins, labels=position_labels)

signal1_table = dataset.groupby('position_tier', observed=True).agg(
    n=('ctr_h1', 'size'),
    mean_ctr_h1=('ctr_h1', 'mean')
).reset_index()

print(signal1_table)

is_monotonic_decreasing = signal1_table['mean_ctr_h1'].is_monotonic_decreasing
print()
print(f"Monotonic decrease across tiers: {is_monotonic_decreasing}")
SIGNAL1_VERDICT = 'CONFIRMED' if is_monotonic_decreasing else 'MIXED'
print(f"VERDICT: {SIGNAL1_VERDICT}")

### 1b. Signal check #2 — volume vs. decline (flag-linked: this is the assumption behind the quick-win flag)

**Hypothesis:** the quick-win flag assumes high-impression pages are worth prioritizing because a fix there recovers more real clicks. Check: does decline rate actually differ by impression volume, or is a high-impression page no more/less likely to be declining than a low-impression one?

In [ ]:
dataset['impressions_tier'] = pd.qcut(dataset['impressions_h1'], q=3, labels=['low', 'mid', 'high'], duplicates='drop')

signal2_table = dataset.groupby('impressions_tier', observed=True).agg(
    n=('decline_label', 'size'),
    decline_rate=('decline_label', 'mean')
).reset_index()

print(signal2_table)

spread = signal2_table['decline_rate'].max() - signal2_table['decline_rate'].min()
print()
print(f"Spread in decline rate across volume tiers: {spread:.1%}")
# Verdict: meaningful spread + high-volume tier NOT lower than low-volume tier -> CONFIRMED
# meaningful spread but in the wrong direction -> OPPOSITE
# negligible spread -> FALSE ; inconsistent/non-monotonic but non-trivial -> MIXED
# --- fill this in by hand after reading the printed table above ---
SIGNAL2_VERDICT = 'MIXED'  # <-- set to CONFIRMED / OPPOSITE / MIXED / FALSE based on what actually printed
print(f"VERDICT: {SIGNAL2_VERDICT}")

**Verdicts:**
- Signal 1 (CTR vs. position): **see `SIGNAL1_VERDICT`** printed above, n per tier shown in `signal1_table`.
- Signal 2 (volume vs. decline): **see `SIGNAL2_VERDICT`** printed above, n per tier shown in `signal2_table` — read the actual spread and direction before locking this in; a `FALSE` or `OPPOSITE` here is a legitimate finding, not a failure, and would mean volume alone should NOT gate the rule below without change.

### 1c. The rule, in plain words

*A page is worth flagging for refresh if it ranks well enough to matter (position tier has a real audience) but its actual CTR is below what pages at that position tier normally get, AND it has enough traffic that fixing it would recover real clicks — not a page nobody sees.*

**Score:** `max(expected_ctr_for_position_tier - ctr_h1, 0) * impressions_h1 * is_visible` — readable, no fitted weights, every term traceable to Section 1a/1b's signals. `expected_ctr_for_position_tier` is the tier's own mean CTR from `signal1_table` above (computed from decision-window data only — not the label, not the outcome window).

**Reason codes (exactly one per row):**
- `ctr_underperform_visible` — score > 0: real CTR gap for its position tier, and enough impressions to matter.
- `low_volume` — CTR gap exists but impressions_h1 is below the median; not worth prioritizing yet.
- `ctr_at_or_above_expected` — no CTR gap; the page is already performing at or above its position tier's norm.

**Action label:** `refresh` when `ctr_underperform_visible`, otherwise `monitor`.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Text answer is in the markdown cells above — the rule itself is encoded in Section 2.
pass

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

In [ ]:
import pandas as pd

# Convert position_tier to string before mapping
expected_ctr_by_tier = (
    signal1_table
    .set_index("position_tier")["mean_ctr_h1"]
    .to_dict()
)

dataset["expected_ctr_tier"] = (
    dataset["position_tier"]
        .astype(str)
        .map(expected_ctr_by_tier)
)

# Convert both columns to numeric
dataset["expected_ctr_tier"] = pd.to_numeric(
    dataset["expected_ctr_tier"],
    errors="coerce"
)

dataset["ctr_h1"] = pd.to_numeric(
    dataset["ctr_h1"],
    errors="coerce"
)

# CTR gap
dataset["ctr_gap"] = (
    dataset["expected_ctr_tier"] -
    dataset["ctr_h1"]
).clip(lower=0)

# Visibility flag
median_impressions = dataset["impressions_h1"].median()

dataset["is_visible"] = (
    dataset["impressions_h1"] >= median_impressions
).astype(int)

# Transparent score
dataset["score"] = (
    dataset["ctr_gap"]
    * dataset["impressions_h1"]
    * dataset["is_visible"]
)

# Reason codes
def reason_code(row):
    if pd.isna(row["expected_ctr_tier"]):
        return "unknown_position_tier"
    elif row["score"] > 0:
        return "ctr_underperform_visible"
    elif row["ctr_gap"] > 0 and row["is_visible"] == 0:
        return "low_volume"
    else:
        return "ctr_at_or_above_expected"

dataset["reason_code"] = dataset.apply(reason_code, axis=1)

# Recommended action
dataset["action"] = dataset["score"].apply(
    lambda s: "refresh" if s > 0 else "monitor"
)

# Rank pages
queue = (
    dataset
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

print(queue["reason_code"].value_counts())
print()
print(queue["action"].value_counts())

queue[
    [
        "client_hash_id",
        "content_hash_id",
        "position_tier",
        "ctr_h1",
        "expected_ctr_tier",
        "impressions_h1",
        "score",
        "reason_code",
        "action",
    ]
].head(10)

In [ ]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_cols = [
    "client_hash_id",
    "content_hash_id",
    "avg_position_h1",
    "position_tier",
    "ctr_h1",
    "expected_ctr_tier",
    "impressions_h1",
    "is_visible",
    "score",
    "reason_code",
    "action",
]

queue[output_cols].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(f"Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")
print("(This file stays out of git by design — CI leak-guard blocks data files. The notebook regenerates it every run.)")

### Precision@K vs. base rate — the number Week 5's model has to beat

`decline_label` is used here ONLY to evaluate the already-built rule, after scoring — it was never an input to the score.

In [ ]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50
base_rate = queue['decline_label'].mean()
p_at_k = precision_at_k(queue['score'].values, queue['decline_label'].values, K)

print(f"Base rate (decline_label mean, whole slice): {base_rate:.1%}")
print(f"Precision@{K} (rule's top {K}):                {p_at_k:.1%}")
print(f"Lift over base rate:                        {p_at_k - base_rate:+.1%}")

In [ ]:
# Save the run's receipts (metrics JSON) — this DOES get committed, unlike the CSV.
import json

metrics = {
    'month': '2026-03',
    'n_rows': int(len(queue)),
    'base_decline_rate': float(base_rate),
    'k': K,
    'precision_at_k': float(p_at_k),
    'signal1_ctr_vs_position_verdict': SIGNAL1_VERDICT,
    'signal2_volume_vs_decline_verdict': SIGNAL2_VERDICT,
    'reason_code_counts': queue['reason_code'].value_counts().to_dict(),
    'action_counts': queue['action'].value_counts().to_dict(),
}

with open('work/outputs/w04_baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

## 3. Top-10 review

*For each of the top 10: the action, why it's there, and what would make it wrong.*

(Assignment card requires ten reviewed rows this week — not twenty; see note at the top of this notebook.)

In [ ]:
top10 = (
    queue.head(10)[
        [
            "client_hash_id",
            "content_hash_id",
            "position_tier",
            "ctr_h1",
            "expected_ctr_tier",
            "impressions_h1",
            "score",
            "reason_code",
            "action",
        ]
    ]
    .rename(
        columns={
            "client_hash_id": "client_id",
            "content_hash_id": "content_id",
        }
    )
    .reset_index(drop=True)
)

top10

**Fill in one line per row after reading the table above** (replace the placeholders once you have real values):

1. `{content_id}` — action **refresh**, reason `ctr_underperform_visible`: position tier `{tier}` but CTR `{ctr_h1}` vs. expected `{expected_ctr_tier}` for that tier, with `{impressions_h1}` impressions. Would be wrong if: this content's tier-average CTR is being dragged down by a handful of outlier pages, making `{tier}`'s own "expected" number an unreliable yardstick for this specific page.
2. *(repeat for rows 2-10, one line each, using each row's real numbers — do not copy row 1's reasoning verbatim.)*

*Do this by hand against the printed `top10` table — the specific numbers must come from your own run, not placeholder text.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Rename dataset keys to match top10
dataset_for_merge = dataset.rename(
    columns={
        "client_hash_id": "client_id",
        "content_hash_id": "content_id",
    }
)

weak_candidates = top10.merge(
    dataset_for_merge[
        [
            "client_id",
            "content_id",
            "days_with_impressions_h1",
        ]
    ],
    on=["client_id", "content_id"],
    how="left",
)

weak_candidates["thin_evidence"] = (
    weak_candidates["days_with_impressions_h1"] < 10
)

print(
    weak_candidates[
        [
            "content_id",
            "days_with_impressions_h1",
            "thin_evidence",
            "score",
        ]
    ]
)

print()
print(
    f"Weak picks (thin evidence in top 10): "
    f"{weak_candidates['thin_evidence'].sum()} of {len(weak_candidates)}"
)

In [ ]:
# Leakage check: confirm the RULE's inputs contain nothing from the outcome window or the label.
rule_inputs = {'position_tier', 'ctr_h1', 'expected_ctr_tier', 'impressions_h1', 'is_visible', 'ctr_gap'}
forbidden = {'clicks_h2', 'decline_label'}

leak_found = rule_inputs & forbidden
assert not leak_found, f"LEAKAGE: rule inputs include outcome-window/label columns: {leak_found}"
print('Leakage check passed: rule inputs are', sorted(rule_inputs))
print('Confirmed NOT in rule inputs (outcome-window / label only):', sorted(forbidden))
print()
print('decline_label and clicks_h2 appear only in the precision@K evaluation cell above, after scoring — never in the score itself.')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

> `work/outputs/baseline_action_score.csv` regenerates on every run and is NOT committed (leak-guard blocks data files).
> `work/outputs/w04_baseline_metrics.json` IS committed — it's this run's receipts (verdicts, precision@K, base rate).
> Run top to bottom in Colab, let `SIGNAL2_VERDICT` reflect what actually printed (it's a placeholder above), and fill in the Section 3 top-10 lines with your real numbers before committing.